In [1]:
import networkx as nx
import pandas as pd
import plotly.express as px

edges_df = pd.read_parquet('../data/frenet_edges.parquet')

G = nx.DiGraph()
for _, row in edges_df.iterrows():
    G.add_edge(row['from_state_id'], row['to_state_id'], weight=row['cost'])

start_node = 0
end_node = 5593

shortest_path = nx.shortest_path(G, source=start_node, target=end_node, weight='weight')
total_cost = nx.shortest_path_length(G, source=start_node, target=end_node, weight='weight')

edges_df_lookup = edges_df.set_index(['from_state_id', 'to_state_id'])

total_m_H2 = 0
total_t = 0
path_rows = []

print("\n--- Edge Transitions ---")
for i in range(len(shortest_path) - 1):
    u = shortest_path[i]
    v = shortest_path[i + 1]

    row = edges_df_lookup.loc[(u, v)]
    acc = row['acceleration_long']
    cost = row['cost']
    m_H2 = row['m_H2']
    vi = row['from_v']
    t = row['time_sec']
    total_m_H2 += m_H2
    total_t += t

    path_rows.append(row)

    print(f"Edge {u} → {v} | Acc: {acc:.3f} | Cost: {cost:.3f} | m_H2: {m_H2:.3f} | v: {vi:.3f}")

print("\n--- Summary ---")
print("Path Length:", len(shortest_path))
print("Total Cost:", total_cost)
print(f"Total Fuel Consumed (m_H2): {total_m_H2}")
print(f"Total Time (s): {total_t:.6f}")

path_df = pd.DataFrame(path_rows)

plot_df = path_df[["from_E", "from_N", "from_s", "from_d", "from_v"]].copy()
plot_df.loc[len(plot_df)] = [
    path_df["to_E"].iloc[-1],
    path_df["to_N"].iloc[-1],
    path_df["to_s"].iloc[-1],
    path_df["to_d"].iloc[-1],
    path_df["to_v"].iloc[-1],
]

export_df = plot_df.rename(columns={"from_E": "E", "from_N": "N", "from_s": "s", "from_d": "d", "from_v": "v"})
export_df.to_csv("../data/trajectory_points.csv", index=False)
print("CSV exported as 'trajectory_points.csv'")

cdf = pd.read_csv('../data/track_centerline.csv')

fig = px.scatter(
    cdf,
    x="E",
    y="N",
    opacity=0.3,
    labels={"E": "Easting (m)", "N": "Northing (m)"},
    title="Vehicle Trajectory on Track"
)

fig.add_traces(
    px.scatter(
        plot_df,
        x="from_E",
        y="from_N",
        color="from_v",
        color_continuous_scale="gray",
        title="Vehicle Trajectory (Velocity Gradient)",
        labels={"from_E": "Easting (m)", "from_N": "Northing (m)", "from_v": "Velocity (m/s)"},
    ).update_traces(mode="lines+markers", line=dict(width=3)).data
)

fig.update_layout(
    xaxis=dict(scaleanchor="y", scaleratio=1),
    template="plotly_dark"
)

fig.show()


--- Edge Transitions ---
Edge 0 → 12.0 | Acc: 0.710 | Cost: 0.000 | m_H2: 0.000 | v: 0.000
Edge 12.0 → 40.0 | Acc: -0.043 | Cost: 0.000 | m_H2: 0.000 | v: 7.237
Edge 40.0 → 70.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 70.0 → 100.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 100.0 → 130.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 130.0 → 160.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 160.0 → 190.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 190.0 → 220.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 220.0 → 250.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 250.0 → 280.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 280.0 → 310.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 310.0 → 340.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 340.0 → 370.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 | v: 7.058
Edge 370.0 → 400.0 | Acc: 0.000 | Cost: 0.000 | m_H2: 0.000 